# Object Detection and Classification Pipeline with YOLO and OpenVINO™

This notebook demonstrates how to build a complete object detection and classification pipeline using YOLO models with OpenVINO™. The pipeline includes:

1. Object detection using YOLOv11
2. Cropping detected objects
3. Classification of cropped objects using YOLO classification model
4. Performance comparison across different devices (CPU, GPU, NPU)

The notebook showcases device selection capabilities and demonstrates how to optimize inference performance by running different parts of the pipeline on different hardware accelerators.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Imports](#Imports)
- [Download Models](#Download-Models)
- [Basic Inference without OpenVINO](#Basic-Inference-without-OpenVINO)
- [Convert to OpenVINO Format](#Convert-to-OpenVINO-Format)
    - [Convert Detection Model](#Convert-Detection-Model)
    - [Convert Classification Model](#Convert-Classification-Model)
- [Select Inference Device](#Select-Inference-Device)
- [Run Object Detection](#Run-Object-Detection)
- [Extract Detected Objects](#Extract-Detected-Objects)
- [Classify Detected Objects](#Classify-Detected-Objects)
- [Complete Pipeline](#Complete-Pipeline)
- [Performance Comparison](#Performance-Comparison)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/yolo-detection-classification/yolo-detection-classification.ipynb" />
### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites

[back to top ⬆️](#Table-of-contents:)

Install required packages.

In [ ]:
%pip install -q "openvino>=2023.1.0" "ultralytics==8.3.0" opencv-python matplotlib Pillow ipywidgets --extra-index-url https://download.pytorch.org/whl/cpu

## Imports

[back to top ⬆️](#Table-of-contents:)

In [ ]:
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import openvino as ov
import torch
from PIL import Image
from ultralytics import YOLO
from IPython.display import display, HTML

# Fetch `notebook_utils` module
import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

from notebook_utils import device_widget

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("yolo-detection-classification.ipynb")

## Download Models

[back to top ⬆️](#Table-of-contents:)

This tutorial uses YOLOv11n for object detection and YOLOv11n-cls for classification. The models will be automatically downloaded from Ultralytics.

In [ ]:
# Create models directory
models_dir = Path("./models")
models_dir.mkdir(exist_ok=True)

# Define model names
DET_MODEL_NAME = "yolo11n"
CLASS_MODEL_NAME = "yolo11n-cls"

# Download detection model
det_model = YOLO(models_dir / f"{DET_MODEL_NAME}.pt")
label_map = det_model.model.names

print(f"Detection model downloaded: {DET_MODEL_NAME}")
print(f"Number of classes: {len(label_map)}")

## Basic Inference without OpenVINO

[back to top ⬆️](#Table-of-contents:)

First, let's run object detection using the PyTorch model to establish a baseline.

In [ ]:
# Define test image path
IMAGE_PATH = Path("grocery.jpeg")

# Run inference with PyTorch model
res = det_model(IMAGE_PATH)

# Display results
Image.fromarray(res[0].plot()[:, :, ::-1])

## Convert to OpenVINO Format

[back to top ⬆️](#Table-of-contents:)

To benefit from OpenVINO optimizations and device flexibility, we need to convert both models to OpenVINO IR format.

### Convert Detection Model

[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Convert detection model to OpenVINO format
det_model_path = models_dir / f"{DET_MODEL_NAME}_openvino_model/{DET_MODEL_NAME}.xml"

if not det_model_path.exists():
    # Export the model to OpenVINO format
    det_model.export(format="openvino", dynamic=True, half=True)
    print(f"Detection model exported to {det_model_path}")
else:
    print(f"Detection model already exists at {det_model_path}")

# Initialize OpenVINO Core
core = ov.Core()

# Read the detection model
det_ov_model = core.read_model(det_model_path)
print(f"\nDetection model input shape: {det_ov_model.input().partial_shape}")
print(f"Detection model output shape: {det_ov_model.output().partial_shape}")

### Convert Classification Model

[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Download classification model
class_model = YOLO(models_dir / f"{CLASS_MODEL_NAME}.pt")

# Convert classification model to OpenVINO format
class_model_path = models_dir / f"{CLASS_MODEL_NAME}_openvino_model/"

if not class_model_path.exists():
    class_model.export(format="openvino")
    print(f"Classification model exported to {class_model_path}")
else:
    print(f"Classification model already exists at {class_model_path}")

## Select Inference Device

[back to top ⬆️](#Table-of-contents:)

Select the device for running object detection inference. Available options include CPU, GPU, and NPU (if available on your system).

In [ ]:
detect_device = device_widget()

detect_device

## Run Object Detection

[back to top ⬆️](#Table-of-contents:)

Perform object detection using the OpenVINO model on the selected device.

In [ ]:
# Compile the detection model for the selected device
ov_config = {}
if detect_device.value != "CPU":
    det_ov_model.reshape({0: [1, 3, 640, 640]})
    if "GPU" in detect_device.value:
        ov_config = {"GPU_DISABLE_WINOGRAD_CONVOLUTION": "YES"}

det_compiled_model = core.compile_model(det_ov_model, detect_device.value, ov_config)


# Set up the YOLO model to use OpenVINO for inference
def infer(*args):
    result = det_compiled_model(args)
    return torch.from_numpy(result[0])


det_model.predictor.inference = infer
det_model.predictor.model.pt = False

# Perform detection
detect_res = det_model(IMAGE_PATH)

# Print inference time
r = detect_res[0]
if hasattr(r, "speed") and r.speed is not None:
    inference_time_ms = r.speed.get("inference", float("nan"))
    print(f"Detection Inference Time: {inference_time_ms:.2f} ms")

# Display detection results
Image.fromarray(detect_res[0].plot()[:, :, ::-1])

## Extract Detected Objects

[back to top ⬆️](#Table-of-contents:)

Extract and save cropped images of detected objects for classification.

In [ ]:
# Load the original image
image = cv2.imread(str(IMAGE_PATH))

# Extract cropped images from detections
cropped_images = []
confidence_threshold = 0.5

for result in detect_res:
    boxes = result.boxes
    if len(boxes) > 0:
        for box in boxes:
            confidence = box.conf.item()
            if confidence > confidence_threshold:
                # Get bounding box coordinates
                x1, y1, x2, y2 = box.xyxy[0]
                # Crop the image
                cropped_image = image[int(y1) : int(y2), int(x1) : int(x2)]
                cropped_images.append(cropped_image)

print(f"Extracted {len(cropped_images)} objects from the image")

### Visualize Cropped Objects

[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Display cropped images
for i, cropped_image in enumerate(cropped_images[:5]):  # Show first 5 for brevity
    # Convert BGR to RGB
    rgb_image = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2RGB)

    # Create a new figure for each image
    plt.figure(figsize=(4, 4))
    plt.imshow(rgb_image)
    plt.title(f"Detected Object {i+1}")
    plt.axis("off")
    plt.show()

## Classify Detected Objects

[back to top ⬆️](#Table-of-contents:)

Now, let's classify each detected object using the YOLO classification model.

### Select Classification Device

[back to top ⬆️](#Table-of-contents:)

In [ ]:
classify_device = device_widget()

classify_device

### Run Classification

[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Load the classification model
class_ov_model = YOLO(str(class_model_path))

# Determine device string for Ultralytics
if classify_device.value == "GPU":
    class_device_to_use = "intel:gpu"
else:
    class_device_to_use = "CPU"

# Classify each cropped object
if not cropped_images:
    print("No cropped images to classify.")
else:
    print(f"Classifying {len(cropped_images)} objects...\n")

    for i, img_data in enumerate(cropped_images):
        try:
            # Run classification
            class_results = class_ov_model.predict(source=img_data, conf=0.5, imgsz=224, device=class_device_to_use, verbose=False)

            if class_results:
                r = class_results[0]

                if hasattr(r, "probs") and r.probs is not None:
                    top1_index = r.probs.top1
                    top1_confidence = r.probs.top1conf.item()
                    class_names = r.names
                    top_class_name = class_names[top1_index]

                    print(f"Object {i+1}: {top_class_name} (Confidence: {top1_confidence:.4f})")

                    # Print inference time if available
                    if hasattr(r, "speed") and r.speed is not None:
                        inference_time_ms = r.speed.get("inference", float("nan"))
                        print(f"  Inference Time: {inference_time_ms:.2f} ms\n")
                else:
                    print(f"Object {i+1}: No classification results\n")

        except Exception as e:
            print(f"Error classifying object {i+1}: {e}\n")

## Complete Pipeline

[back to top ⬆️](#Table-of-contents:)

Now let's combine detection and classification into a complete pipeline and measure its performance.

### Select Devices for Pipeline

[back to top ⬆️](#Table-of-contents:)

In [ ]:
pipeline_detect_device = device_widget()
pipeline_classify_device = device_widget()

display(pipeline_detect_device, pipeline_classify_device)

### Run Complete Pipeline with Performance Measurement

[back to top ⬆️](#Table-of-contents:)

In [ ]:
# Prepare Detection Model
ov_config = {}
if pipeline_detect_device.value != "CPU":
    det_ov_model.reshape({0: [1, 3, 640, 640]})
    if "GPU" in pipeline_detect_device.value:
        ov_config = {"GPU_DISABLE_WINOGRAD_CONVOLUTION": "YES"}

det_compiled_model = core.compile_model(det_ov_model, pipeline_detect_device.value, ov_config)


# Update inference function
def infer(*args):
    result = det_compiled_model(args)
    return torch.from_numpy(result[0])


det_model.predictor.inference = infer
det_model.predictor.model.pt = False

# Prepare Classification Model
class_ov_model = YOLO(str(class_model_path))

if pipeline_classify_device.value == "GPU":
    pipeline_class_device_to_use = "intel:gpu"
    # Warm up GPU by running a dummy prediction
    dummy_img = Image.new("RGB", (224, 224), color="red")
    dummy_img_np = np.array(dummy_img)
    class_ov_model.predict(source=dummy_img_np, conf=0.5, imgsz=224, device=pipeline_class_device_to_use, verbose=False)
else:
    pipeline_class_device_to_use = "CPU"

# Start Timer
start_time = time.perf_counter()

# Step 1: Object Detection
detect_res = det_model(IMAGE_PATH)

# Step 2: Extract Cropped Images
image = cv2.imread(str(IMAGE_PATH))
cropped_images = []
confidence_threshold = 0.5

for result in detect_res:
    boxes = result.boxes
    if len(boxes) > 0:
        for box in boxes:
            confidence = box.conf.item()
            if confidence > confidence_threshold:
                x1, y1, x2, y2 = box.xyxy[0]
                cropped_image = image[int(y1) : int(y2), int(x1) : int(x2)]
                cropped_images.append(cropped_image)

# Step 3: Classification
if cropped_images:
    for i, img_data in enumerate(cropped_images):
        try:
            class_results = class_ov_model.predict(source=img_data, conf=0.5, imgsz=224, device=pipeline_class_device_to_use, verbose=False)
        except Exception as e:
            print(f"Error during classification of object {i+1}: {e}")

# End Timer
end_time = time.perf_counter()
elapsed_time = end_time - start_time

# Print Results
print(f"\nNumber of objects detected: {len(cropped_images)}")
print(f"Total Pipeline Time: {elapsed_time:.4f} seconds ({elapsed_time*1000:.2f} ms)")
print(f"Detection Device: {pipeline_detect_device.value}")
print(f"Classification Device: {pipeline_classify_device.value}")

## Performance Comparison

[back to top ⬆️](#Table-of-contents:)

Display the pipeline performance in a formatted way.

In [ ]:
# Display formatted results
display(HTML("<h3 style='color: blue;'>Pipeline Performance Summary</h3>"))
display(HTML(f"<p style='font-size: 18px;'><b>Detection Device:</b> <span style='color: orange;'>{pipeline_detect_device.value}</span></p>"))
display(HTML(f"<p style='font-size: 18px;'><b>Classification Device:</b> <span style='color: orange;'>{pipeline_classify_device.value}</span></p>"))
display(HTML(f"<p style='font-size: 18px;'><b>Objects Detected:</b> <span style='color: purple;'>{len(cropped_images)}</span></p>"))
display(HTML(f"<p style='font-size: 20px;'><b>Total Pipeline Time:</b> <span style='color: green; font-weight: bold;'>{elapsed_time*1000:.2f} ms</span></p>"))

## Conclusion

[back to top ⬆️](#Table-of-contents:)

This notebook demonstrated how to:
1. Convert YOLO models to OpenVINO format
2. Run object detection and classification on different devices
3. Build a complete detection and classification pipeline
4. Measure and compare performance across devices

By leveraging OpenVINO, you can achieve better performance and flexibility in deploying your computer vision models across various hardware platforms.